In [2]:
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

# Big(ish) Data Handling

## Data Storage, Data Formats, Out-of-core processing and Data Accessing Strategies

### Learning Outcomes

At the end of this lecture, you should be able to:

* work with 'inconveniently large' datasets
* chunk large datasets 
* convert datasets to HDF5 
* perform out-of-core processing
* optimise dataset storage
* interface with sqlite datasets



This notebook will use the NYC Taxi Trip 2013-2014 dataset made up of 12 files, one for each month. The complete dataset is about 31GB. In the interest of time during class, the examples here will only use a portion of the data from the month of January. You are encouraged to employ the strategies below on the entire dataset in your own time. 

Dataset and selected examples drawn from http://www.andresmh.com/nyctaxitrips/. More info on the NYC Taxi Trip 2013-2014 dataset can be found here http://chriswhong.com/open-data/foil_nyc_taxi/ and http://hafen.github.io/taxi/#background. Selected material on Dask usage, sourced from https://github.com/dask/dask-tutorial 

#### Massey Lab Instructions

1. download trip_data_1a.csv and have it ready for class (especially if using a laptop)
2. save your file in C:/I folder because your H drive is likely to run out of space
3. write all files to the C drive for the same reason as above

In [9]:
# !pip install polars
# !pip install duckdb
# !pip install parquet


------------------------

In [1]:
!pip install ipython_memwatcher

  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for ipython_memwatcher: filename=ipython_memwatcher-0.2.5-py3-none-any.whl size=3596 sha256=07a4452ef69d3224961d7a3067815849ad5ea5d4946bf7f5626565561b636b31
  Stored in directory: c:\users\johnl\appdata\local\pip\cache\wheels\27\f5\f1\bd6dec4fbdf4f7ce7d5ddc1ab98ebc01e87d0c1c0242290ad1
Successfully built ipython_memwatcher

   ---------------------------------------- 2/2 [ipython_memwatcher]



  DEPRECATION: Building 'ipython_memwatcher' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'ipython_memwatcher'. Discussion can be found at https://github.com/pypa/pip/issues/6334


In [3]:
from ipython_memwatcher import MemWatcher
mw = MemWatcher()
mw.start_watching_memory()

Error in callback <bound method MemWatcher.pre_run_cell of <ipython_memwatcher.memwatcher.MemWatcher object at 0x000002651D1B3B60>> (for pre_run_cell), with arguments args (<ExecutionInfo object at 2651f71eb10, raw_cell="from ipython_memwatcher import MemWatcher
mw = Mem.." store_history=True silent=False shell_futures=True cell_id=49cf3b55-428d-4ff6-91a8-0cd7abe414a5>,),kwargs {}:


TypeError: MemWatcher.pre_run_cell() takes 1 positional argument but 2 were given

Error in callback <bound method MemWatcher.watch_memory of <ipython_memwatcher.memwatcher.MemWatcher object at 0x000002651D1B3B60>> (for post_run_cell), with arguments args (<ExecutionResult object at 2651f71ec40, execution_count=3 error_before_exec=None error_in_exec=None info=<ExecutionInfo object at 2651f71eb10, raw_cell="from ipython_memwatcher import MemWatcher
mw = Mem.." store_history=True silent=False shell_futures=True cell_id=49cf3b55-428d-4ff6-91a8-0cd7abe414a5> result=None>,),kwargs {}:


TypeError: MemWatcher.watch_memory() takes 1 positional argument but 2 were given

Error in callback <bound method MemWatcher.watch_memory of <ipython_memwatcher.memwatcher.MemWatcher object at 0x000002651BACDE50>> (for post_run_cell), with arguments args (<ExecutionResult object at 2651f71ec40, execution_count=3 error_before_exec=None error_in_exec=None info=<ExecutionInfo object at 2651f71eb10, raw_cell="from ipython_memwatcher import MemWatcher
mw = Mem.." store_history=True silent=False shell_futures=True cell_id=49cf3b55-428d-4ff6-91a8-0cd7abe414a5> result=None>,),kwargs {}:


TypeError: MemWatcher.watch_memory() takes 1 positional argument but 2 were given

> Memwatcher: https://github.com/FrancescAlted/ipython_memwatcher

# Limitations of Pandas 

According to Wes McKinney, the author of Python Pandas, there are some serious limitations inherent in Pandas when it comes to dealing with larger datasets that were not considered when the project to develop Pandas was initiated:

> To put it simply, we weren't thinking about analyzing 100 GB or 1 TB datasets in 2011. Nowadays, my rule of thumb for pandas is that you should have 5 to 10 times as much RAM as the size of your dataset. So if you have a 10 GB dataset, you should really have about 64, preferably 128 GB of RAM if you want to avoid memory management problems. This comes as a shock to users who expect to be able to analyze datasets that are within a factor of 2 or 3 the size of their computer's RAM.

>> **pandas rule of thumb: have 5 to 10 times as much RAM as the size of your dataset**

> There are additional, hidden memory killers in the project, like the way that we use Python objects (like strings) for many internal details, so **it's not unusual to see a dataset that is 5GB on disk take up 20GB or more in memory**. It's an overall bad situation for large datasets.

Source: http://wesmckinney.com/blog/apache-arrow-pandas-internals/

# Splitting large files into smaller chunks - the naive approach



In [87]:
# SET YOUR FILE PATH TO WHERE YOUR DATASET IS STORED
file_path = 'trip_data_1a.zip'

In [2]:
import pandas as pd
import numpy as np
import datetime as dt


Let's assume that the file we are dealing with in this example cannot fit into out working memory (RAM). What can we do?....

Well, given a very large csv file, often the most simplistic approach to working with it is to split it into multiple smaller csv file chunks, and then process each one in turn.

Here we will re-write each chunk to disk.

In [4]:
import zipfile
z = zipfile.ZipFile('trip_data_1a.zip')


In [5]:

# the original trip_data_1.csv data file has over 14M records - 
# the trip_data_1a.csv has 5M rows which we will divide here into three chunks
chunksize = 2000000
i = 0
f = ['a','b','c']
for df in pd.read_csv(z.open('trip_data_1a.csv'), chunksize=chunksize):
        print('writing: ', file_path + 'trip_data_1a_' + f[i] + '.csv')
        df.to_csv(file_path + 'trip_data_1a_' + f[i] + '.csv', index=None)
        i += 1

writing:  trip_data_1a.csvtrip_data_1a_a.csv
writing:  trip_data_1a.csvtrip_data_1a_b.csv
writing:  trip_data_1a.csvtrip_data_1a_c.csv


OR

In [ ]:
# the original trip_data_1.csv data file has over 14M records - 
# the trip_data_1a.csv has 5M rows which we will divide here into three chunks
chunksize = 2000000
i = 0
f = ['a','b','c']
for df in pd.read_csv(file_path + 'trip_data_1a.csv', chunksize=chunksize):
        df.to_csv(file_path + 'trip_data_1a_' + f[i] + '.csv', index=None)
        i += 1

# Optimising Pandas DataFrame Accesses 

When performing heavily intensive computations on dataframes of a large size, it is imperative that some form of profiling and optimisation is carried out.

The way we process and access pandas dataframes can have significant effects on the runtime performance that differs in orders of magnitude between various options.  

Since we can fit all our data into memory, let's do this in order to explore some pandas performance characteristics.

http://jose-coto.com/slicing-methods-pandas
https://stackoverflow.com/questions/28757389/loc-vs-iloc-vs-ix-vs-at-vs-iat

In [85]:
#read it as a zipped file or a csv
df = pd.read_csv(z.open('trip_data_1a.csv'), infer_datetime_format=True, parse_dates=['pickup_datetime', 'dropoff_datetime'])

C:\Users\johnl\AppData\Local\Temp\ipykernel_21196\3774988069.py:2: FutureWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df = pd.read_csv(z.open('trip_data_1a.csv'), infer_datetime_format=True, parse_dates=['pickup_datetime', 'dropoff_datetime'])


OR

In [14]:
# df = pd.read_csv(file_path + 'trip_data_1a.csv', infer_datetime_format=True, parse_dates=['pickup_datetime', 'dropoff_datetime'])

In [15]:
df.head()

,medallion,hack_license,vendor_id,rate_code,store_and_fwd_flag,pickup_datetime,dropoff_datetime,passenger_count,trip_time_in_secs,trip_distance,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude
0,89D227B655E5C82AECF13C3F540D4CF4,BA96DE419E711691B9445D6A6307C170,CMT,1,N,2013-01-01 15:11:48,2013-01-01 15:18:10,4,382,1.0,-73.978165,40.757977,-73.989838,40.751171
1,0BD7C8F5BA12B88E0B67BED28BEA73D8,9FD8F69F0804BDB5549F40E9DA1BE472,CMT,1,N,2013-01-06 00:18:35,2013-01-06 00:22:54,1,259,1.5,-74.006683,40.731781,-73.994499,40.750660
2,0BD7C8F5BA12B88E0B67BED28BEA73D8,9FD8F69F0804BDB5549F40E9DA1BE472,CMT,1,N,2013-01-05 18:49:41,2013-01-05 18:54:23,1,282,1.1,-74.004707,40.737770,-74.009834,40.726002
3,DFD2202EE08F7A8DC9A57B02ACB81FE2,51EE87E3205C985EF8431D850C786310,CMT,1,N,2013-01-07 23:54:15,2013-01-07 23:58:20,2,244,0.7,-73.974602,40.759945,-73.984734,40.759388
4,DFD2202EE08F7A8DC9A57B02ACB81FE2,51EE87E3205C985EF8431D850C786310,CMT,1,N,2013-01-07 23:25:03,2013-01-07 23:34:24,1,560,2.1,-73.976250,40.748528,-74.002586,40.747868


In [16]:
df.tail()

,medallion,hack_license,vendor_id,rate_code,store_and_fwd_flag,pickup_datetime,dropoff_datetime,passenger_count,trip_time_in_secs,trip_distance,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude
4999995,B0B39A24031CFBFACCAFC9B905E6FBC3,D53EB16B85EB24D1E44CD719ABDE2A4B,VTS,1,NaN,2013-01-18 06:40:00,2013-01-18 06:46:00,1,360,1.47,-73.975250,40.752373,-73.988930,40.741718
4999996,1885BADD73092C2F21F220A2F06105CA,646747DDA48B777875121DDEE8CD5C26,VTS,1,NaN,2013-01-18 06:34:00,2013-01-18 06:46:00,1,720,4.49,-73.984795,40.769478,-74.013062,40.717312
4999997,A7E52F824BC1D8DC2535D4C6471E7899,7C653CC679536FAA4C1582017D15023B,VTS,1,NaN,2013-01-17 23:58:00,2013-01-18 00:20:00,1,1320,3.97,-73.990685,40.761143,-73.990318,40.719227
4999998,96748B85E03307579E92AC277D940C8D,633F3FBBB9D435C34820C1CB3B752AEC,VTS,1,NaN,2013-01-18 00:15:00,2013-01-18 00:18:00,1,180,0.35,-73.976257,40.755478,-73.972755,40.753647
4999999,A8EE7FD7022C9E58BE8F096D347EE0B8,164B133FF8E8A8C7AE323DB990B37ADF,VTS,1,NaN,2013-01-17 23:51:00,2013-01-18 00:13:00,6,1320,11.19,-73.993210,40.733463,-73.867737,40.824837


In [17]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000000 entries, 0 to 4999999
Data columns (total 14 columns):
 #   Column              Dtype         
---  ------              -----         
 0   medallion           object        
 1   hack_license        object        
 2   vendor_id           object        
 3   rate_code           int64         
 4   store_and_fwd_flag  object        
 5   pickup_datetime     datetime64[ns]
 6   dropoff_datetime    datetime64[ns]
 7   passenger_count     int64         
 8   trip_time_in_secs   int64         
 9   trip_distance       float64       
 10  pickup_longitude    float64       
 11  pickup_latitude     float64       
 12  dropoff_longitude   float64       
 13  dropoff_latitude    float64       
dtypes: datetime64[ns](2), float64(5), int64(3), object(4)
memory usage: 534.1+ MB


In [ ]:
#df = pd.read_hdf('../datasets/trip_data_1a.h5', '/data', start=0, stop=1000000)
#df.tail()

We will consider looping through a dataframe that has 1M rows.

iat - position based works similarly to iloc.

Advantage over iloc is that this is faster.

Disadvantage is that you can't use arrays for indexers. Cannot assign new indices and columns.

In [18]:
import random

random.randint(0, 4999999)

3625643

In [19]:
def dataframe_access_using_iat(df):
    count = 0
    while count < 1000000: 
        df.iat[random.randint(0, 4999999), 0]
        count += 1
        
%time dataframe_access_using_iat(df)

CPU times: total: 15 s
Wall time: 15.1 s


In [20]:
# example
df.iat[1000000, 0] = 0

In [31]:
# example
df.iat[ 1, 0]

'0BD7C8F5BA12B88E0B67BED28BEA73D8'

In [34]:
# example
df.iat[1, 0]

'0BD7C8F5BA12B88E0B67BED28BEA73D8'

at - label based works very similarly to loc for scalar indexers.

Advantage over loc is that this is faster. 

Disadvantage is that you can't use arrays for indexers. Cannot assign new indices and columns.

In [35]:
def dataframe_access_using_at(df):
    count = 0
    while count < 1000000: 
        df.at[random.randint(0, 4999999), 'medallion']
        count += 1

%time dataframe_access_using_at(df)

CPU times: total: 3.48 s
Wall time: 3.53 s


In [36]:
# example
df.at[0, 0]

KeyError: 0

In [37]:
# example
df.at[10000000, 'medallion'] = 0

In [41]:
df.at[1, 'medallion']

'0BD7C8F5BA12B88E0B67BED28BEA73D8'

iloc - position based
Similar to loc except with positions rather that index values. However, you cannot assign new columns or indices.

In [42]:
def dataframe_access_using_iloc(df):
    count = 0
    while count < 1000000: 
        df.iloc[random.randint(0, 4999999), 0]
        count += 1
    

%time dataframe_access_using_iloc(df)

CPU times: total: 15.4 s
Wall time: 15.5 s


In [43]:
# example
df.iloc[ [0,1] , 0] 

0    89D227B655E5C82AECF13C3F540D4CF4
1    0BD7C8F5BA12B88E0B67BED28BEA73D8
Name: medallion, dtype: object

In [44]:
# example
df.iloc[ [0,1], 'medallion']

ValueError: Location based indexing can only have [integer, integer slice (START point is INCLUDED, END point is EXCLUDED), listlike of integers, boolean array] types

**Exercise:** *.ix* operator is now deprecated but the *.loc* access operator still remains as untested. Using the approach above, create a test for this operator below.

loc - label based
Allows you to pass 1-D arrays as indexers. Arrays can be either slices (subsets) of the index or column, or they can be boolean arrays which are equal in length to the index or columns.

Special Note: when a scalar indexer is passed, loc can assign a new index or column value that didn't exist before.

In [67]:
# example
df.loc[ [0,1] , 'medallion']

0    89D227B655E5C82AECF13C3F540D4CF4
1    0BD7C8F5BA12B88E0B67BED28BEA73D8
Name: medallion, dtype: object

In [66]:
def dataframe_access_using_loc(df):
    count = 0
    while count < 1000000: 
        df.loc[random.randint(0, 4999999), 'medallion']
        count += 1
    

%time dataframe_access_using_loc(df)

CPU times: total: 9.72 s
Wall time: 9.78 s


# Strategies for Processing Large Datasets 

Most python modules and machine learning algorithms assume that all the data can be loaded into memory.

This is increasingly becoming unrealistic with the amount of data that is being generated, stored and used for data analysis.

This creates considerable challenges. When the data is in the order of terabytes, then Hadoop distributed processing on clusters of commodity machines is the standard approach.

Such infrastructure is not always available. 

When the datasets are in tens of gigabytes in size, Hadoop is an overkill and is often not the preferred option. 

Data that is larger than the size of a machine's RAM capacity is not Big Data, but it is large and inconvenient to work with. The challenges with this kind of data are:

1. execution runtime
2. working with libraries that enable processing the data without bringing all of it into RAM (out-of-core processing)
3. ensuring that dataset file formats are optimised for out-of-core processing

First of all, since all of our data fits into the working memory, we can perform pretty fast computations on millions of records since the CPU is not having to access the disk.

In [68]:
%time df.passenger_count.mean()

CPU times: total: 31.2 ms
Wall time: 29.6 ms


np.float64(1.679133)

In [69]:
%time df.trip_time_in_secs.mean()

CPU times: total: 31.2 ms
Wall time: 28.4 ms


np.float64(694.4157456)

In [70]:
%time df.describe()

CPU times: total: 2.02 s
Wall time: 2.06 s


,rate_code,pickup_datetime,dropoff_datetime,passenger_count,trip_time_in_secs,trip_distance,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude
count,5.000000e+06,5000000,5000000,5.000000e+06,5.000000e+06,5.000000e+06,5.000000e+06,5.000000e+06,4.999962e+06,4.999962e+06
mean,1.040434e+00,2013-01-10 09:30:44.493574400,2013-01-10 09:42:20.371260416,1.679133e+00,6.944157e+02,2.945738e+00,-7.264334e+01,4.001542e+01,-7.261038e+01,3.999927e+01
min,0.000000e+00,2013-01-01 00:00:00,2013-01-01 00:02:00,0.000000e+00,0.000000e+00,0.000000e+00,-2.134648e+03,-3.447918e+03,-2.331833e+03,-3.255573e+03
25%,1.000000e+00,2013-01-04 16:35:00,2013-01-04 16:47:00,1.000000e+00,3.600000e+02,1.100000e+00,-7.399216e+01,4.073458e+01,-7.399151e+01,4.073371e+01
50%,1.000000e+00,2013-01-10 18:46:40,2013-01-10 18:59:06,1.000000e+00,5.810000e+02,1.800000e+00,-7.398189e+01,4.075238e+01,-7.398046e+01,4.075281e+01
75%,1.000000e+00,2013-01-15 21:19:00,2013-01-15 21:29:59,2.000000e+00,9.000000e+02,3.250000e+00,-7.396722e+01,4.076679e+01,-7.396423e+01,4.076777e+01
max,2.100000e+02,2013-01-25 07:45:22,2013-01-25 08:01:07,2.080000e+02,1.080000e+04,1.000000e+02,8.084212e+01,3.124198e+03,1.114939e+02,3.477106e+03
std,3.450952e-01,NaN,NaN,1.351074e+00,4.903858e+02,3.430111e+00,1.007255e+01,7.936117e+00,1.018828e+01,7.981045e+00


# Polars

## What is Polars?

Polars is a high-performance DataFrame library written in Rust. It is designed to handle large datasets efficiently by leveraging multicore processing and lazy execution. Unlike traditional libraries such as pandas, Polars is optimised for speed and memory usage, making it a great choice for performance-critical applications. Its growing popularity, driven by a focus on scalability and computational efficiency, has attracted a vibrant community of users and contributors.

## Why Polars?

If you have a very large dataset that can fit into memory, then Polars will provide some additional performance efficiency when needed.

- **Speed**: Polars is faster than pandas for many operations because it is implemented in Rust and uses parallelisation effectively.
- **Multi-threading**: Polars efficiently uses all available CPU cores, making it ideal for large datasets and complex computations.
- **Lazy Execution**: Polars defers computations until necessary, optimising the execution plan and reducing unnecessary operations.
- **Memory Efficiency**: Rust’s ownership model and zero-cost abstractions allow Polars to handle memory more efficiently than pandas.

### Columnar Memory Layout vs Row-Wise Memory Layout

Understanding how data is stored in memory is crucial for optimising performance in data-intensive applications. Two common memory layouts used in data processing are columnar and row-wise memory layouts. These two approaches are optimised for different types of data access patterns and can have significant effects on performance, particularly for operations on large datasets.

Comparison of Use Cases

    Row-Wise Layout is more suited for transactional workloads where the application frequently reads or writes entire rows. For example, in databases that require frequent inserts, updates, or random row-level access, row-wise memory layouts are optimal.

    Columnar Layout excels in analytical workloads where the application performs operations on a subset of columns across many rows. For example, when summing the values of a particular column in a large dataset or when filtering rows based on a column's value, columnar storage reduces memory access and speeds up computation.
    

## Memory Efficiency in Polars vs Pandas

### 1. Columnar Data Storage

Polars uses a **columnar memory layout**, where data is stored by column rather than by row. This is more cache-friendly for operations such as aggregations and filtering. It allows for faster access times and better performance, particularly for columnar operations. This is in contrast to the **row-based memory layout** used by pandas, which is less efficient for these types of operations.

### 2. Efficient Use of Rust’s Memory Model

Polars is written in **Rust**, which offers a more efficient memory management model compared to Python’s garbage collection. Rust’s ownership model ensures memory is allocated and deallocated in a predictable manner, leading to fewer memory leaks and reduced overhead. This allows Polars to manage memory more efficiently while retaining high performance.

### 3. Arrow Format Compatibility

Polars uses the **Apache Arrow** format under the hood, which is optimised for efficient columnar data storage in memory. Arrow’s lightweight, standardised memory formats enable faster processing and interoperation with other libraries that use Arrow. This contributes to both lower memory usage and higher performance.

### 4. Compact Data Types

Polars makes use of **compact data types** such as `UInt8`, `Int16`, and `Float32`, which reduce the overall memory footprint. In comparison, pandas defaults to less efficient data types such as `float64` or `object`, which can lead to higher memory consumption.

### 5. Optimised String Handling

Polars handles string data more efficiently by using **categorical encoding** (dictionary encoding). This technique stores each unique string value only once, rather than repeating it for every row, which significantly reduces memory usage. Pandas, on the other hand, stores strings as Python `object` types, which are more memory-intensive.

### 6. Memory Efficient Execution

Polars uses **lazy execution**, meaning it defers computations until absolutely necessary. This allows Polars to build an optimised query plan and execute it all at once, avoiding excessive intermediate memory usage. Pandas, in contrast, uses **eager execution**, which immediately executes operations and can lead to higher memory overhead due to the creation of intermediate objects.

### 7. Avoiding Temporary Objects

Polars minimises the creation of temporary objects during data manipulations, reducing the overall memory overhead. In pandas, operations such as filtering, grouping, or joining often generate multiple intermediate objects, increasing memory consumption. Polars avoids this issue by optimising its internal execution pipeline.




## Weaknesses of Polars

Despite its strengths, Polars is still relatively new compared to pandas. Its ecosystem and integration with machine learning frameworks are less mature, which can make it less convenient for some tasks. However, it is rapidly gaining traction due to its exceptional performance. Also, it does not support out-of-core computation completely.


In [71]:
import polars as pl

In [73]:
# Load the dataset using Polars
df_polars = pl.read_csv('trip_data_1a.csv', try_parse_dates=True) # , parse_dates=['pickup_datetime', 'dropoff_datetime']

df_polars.head()

medallion,hack_license,vendor_id,rate_code,store_and_fwd_flag,pickup_datetime,dropoff_datetime,passenger_count,trip_time_in_secs,trip_distance,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude
str,str,str,i64,str,datetime[μs],datetime[μs],i64,i64,f64,f64,f64,f64,f64
"""89D227B655E5C82AECF13C3F540D4C…","""BA96DE419E711691B9445D6A6307C1…","""CMT""",1,"""N""",2013-01-01 15:11:48,2013-01-01 15:18:10,4,382,1.0,-73.978165,40.757977,-73.989838,40.751171
"""0BD7C8F5BA12B88E0B67BED28BEA73…","""9FD8F69F0804BDB5549F40E9DA1BE4…","""CMT""",1,"""N""",2013-01-06 00:18:35,2013-01-06 00:22:54,1,259,1.5,-74.006683,40.731781,-73.994499,40.75066
"""0BD7C8F5BA12B88E0B67BED28BEA73…","""9FD8F69F0804BDB5549F40E9DA1BE4…","""CMT""",1,"""N""",2013-01-05 18:49:41,2013-01-05 18:54:23,1,282,1.1,-74.004707,40.73777,-74.009834,40.726002
"""DFD2202EE08F7A8DC9A57B02ACB81F…","""51EE87E3205C985EF8431D850C7863…","""CMT""",1,"""N""",2013-01-07 23:54:15,2013-01-07 23:58:20,2,244,0.7,-73.974602,40.759945,-73.984734,40.759388
"""DFD2202EE08F7A8DC9A57B02ACB81F…","""51EE87E3205C985EF8431D850C7863…","""CMT""",1,"""N""",2013-01-07 23:25:03,2013-01-07 23:34:24,1,560,2.1,-73.97625,40.748528,-74.002586,40.747868


In [74]:
%time df_polars.filter(pl.col('passenger_count') > 5)

CPU times: total: 219 ms
Wall time: 41.8 ms


medallion,hack_license,vendor_id,rate_code,store_and_fwd_flag,pickup_datetime,dropoff_datetime,passenger_count,trip_time_in_secs,trip_distance,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude
str,str,str,i64,str,datetime[μs],datetime[μs],i64,i64,f64,f64,f64,f64,f64
"""EE1513D432B07F7E0B5E2ED1EF6290…","""F31D261881520931062C011366E56A…","""VTS""",1,null,2013-01-13 04:30:00,2013-01-13 04:44:00,6,840,5.21,-74.005455,40.740772,-73.967354,40.798096
"""282E88327FA32E0C655B1216A91125…","""7152AE0408719E98C7A42D7B8E9BFB…","""VTS""",1,null,2013-01-13 04:24:00,2013-01-13 04:41:00,6,1020,4.77,-73.988449,40.723179,-73.981247,40.765068
"""FD39403FDE46B6C753DDD6518A4365…","""2B36D07A27BB35D7DF7170C83EEAA1…","""VTS""",1,null,2013-01-13 04:41:00,2013-01-13 04:46:00,6,300,1.25,-73.996262,40.759922,-73.982346,40.765648
"""2789D8398CBD60E51BF7D4BC78F3D7…","""9AB9E0F124D42BC702D0BBCFDA6B68…","""VTS""",1,null,2013-01-13 04:31:00,2013-01-13 04:48:00,6,1020,5.81,-73.996246,40.723808,-73.993622,40.672703
"""A0DEAEC3D5592AE94B876356F12F81…","""6A35995FBD5399AB131B89623C3447…","""VTS""",1,null,2013-01-13 04:37:00,2013-01-13 04:42:00,6,300,1.37,-73.981491,40.741055,-73.978394,40.728966
…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""2C109C7A90FE6F794317484C621B76…","""5017117161B4F130721401AEF80466…","""VTS""",1,null,2013-01-18 00:17:00,2013-01-18 00:31:00,6,840,3.21,-73.987289,40.71846,-73.978264,40.687134
"""050ABCE14371E671A6260A74D2C1F5…","""F0A25E95A68083330E451A69EBA4E6…","""VTS""",1,null,2013-01-18 06:34:00,2013-01-18 06:38:00,6,240,1.29,-73.977608,40.78894,-73.977745,40.77935
"""7A138A34B75F17B789E2B9E9E597FC…","""36E9B6A67E2F57E6D38A66F60563BA…","""VTS""",1,null,2013-01-18 06:39:00,2013-01-18 06:45:00,6,360,1.17,-73.980461,40.726276,-73.992569,40.734249


Polars excels at aggregations, especially with multicore processing. Let’s compute the total trips by vendor.


In [75]:
%time df_polars.group_by(['vendor_id']).agg([ \
    pl.col('passenger_count').count().alias('passenger_count') ])



CPU times: total: 1.61 s
Wall time: 294 ms


vendor_id,passenger_count
str,u32
"""VTS""",2432741
"""CMT""",2567259


Let’s compute the total trip distances by taxi and order by largest value.

In [76]:
%time df_polars.group_by('medallion').agg([ \
    pl.col('trip_distance').sum()]).sort(by='trip_distance', descending=True)

CPU times: total: 1.94 s
Wall time: 385 ms


medallion,trip_distance
str,f64
"""8211BE04462ADE4621B68E1DFEA547…",2710.6
"""6B19837BFEAA374B7B4694799D7CC7…",2140.5
"""C89E044B41A0AF290391EF4D690B0F…",2026.7
"""A0DBCED92075908A8B4A0D8E6420C7…",2026.4
"""6BD1B641A1CD55803A21560299B985…",2023.6
…,…
"""F5143823F1AAC9FC2FB3BD4A48CC56…",0.0
"""0A4F0D017F0B4C75C14F9012661F68…",0.0
"""9BE6AD896FF34E3EC823A5329CAA88…",0.0


**Exercise:** Perform the above using pandas and time the operations and compare.

# My dataset is bigger than my RAM

If this happens and you only have access to the tools covered so far, then you can resort to chunking/splitting the dataset as shown above and working on each part in turn.

This is extremely tedious. Do not do this if you can avoid it. Usually you can.

There are far better strategies available to us. 

**Out-of-core processing** allows us to get around this problem is. However, it can still be very slow when the dataset is stored on disk in a non-binary format like csv. 

Files that are not saved in a **binary compressed format**, are not stored optimally and must also be interpreted as they are read from disk, which slows down the read time. 

Therefore, saving large files in a binary format instead of a text format like a csv, will ensure that the data is stored and read in future in a much more efficient manner.

Saving datasets in specific binary formats also allows us to **filter and read into memory only subsets of data** that meet a required condition, which can significantly limit the RAM requirement. 

## HDF5

**HDF5** (http://www.h5py.org/) is a binary format for efficiently storing and accessing data which makes it much more suitable for sharing and out-of-core processing.

Therefore, when working with very large files, convert and store them in a binary format like HDF5.

HDF5 stands for Hierarchical Data Format. It is a multipurpose hierarchical container format capable of storing large **numerical datasets** with their metadata. The specification is open and the tools are open source. 

It supports an unlimited variety of datatypes, and is designed for flexible and efficient I/O and for high volume and complex data. HDF5 is portable and is extensible.

An HDF5 file contains a hierarchy of numerical arrays (or datasets) organized within groups.

A dataset can be stored in two ways: contiguously or chunked. The former stores a dataset in a contiguous buffer in the file, while the latter splits it uniformly in rectangular chunks organized in a B-tree.

HDF5 also supports lossless compression of datasets.

In [83]:
df = pd.read_csv(file_path + 'trip_data_1a.csv', infer_datetime_format=True, parse_dates=['pickup_datetime', 'dropoff_datetime'])

C:\Users\johnl\AppData\Local\Temp\ipykernel_21196\2201733226.py:1: FutureWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df = pd.read_csv('trip_data_1a.csv', encoding='latin1', infer_datetime_format=True, parse_dates=['pickup_datetime', 'dropoff_datetime'])


ValueError: Missing column provided to 'parse_dates': 'dropoff_datetime, pickup_datetime'

In [88]:
%time df.to_hdf(file_path + 'trip_data_1a.h5', '/data', format='table', mode='w', data_columns=['trip_distance'])

<timed eval>:1: FutureWarning: Starting with pandas version 3.0 all arguments of to_hdf except for the argument 'path_or_buf' will be keyword-only.


CPU times: total: 12 s
Wall time: 12.2 s


#### In the above: 

*format='table'* specifies that the file is searchable

*mode='w' specifies write/create/overwrite 
 
*data_columns=['trip_distance']* cretes an index on the specified column so that we can select/search data on

*/data is the object under which a given dataset is stored and is defined as a file path, you can define this name to whatever is suitable for the domain

In [89]:
hdf = pd.HDFStore(file_path + 'trip_data_1a.h5')
hdf

<class 'pandas.io.pytables.HDFStore'>
File path: trip_data_1a.ziptrip_data_1a.h5

In [90]:
hdf['/data'].shape

(5000000, 14)

Datasets can be appended (using append) to the existing dataset in the HDF5 format, or new objects can be created within the same file under a different group path:

In [91]:
hdf.put('tables/t1', pd.DataFrame(np.random.rand(20,5)))
hdf.put('tables/t2', pd.DataFrame(np.random.rand(10,3)))
hdf.put('data/t1', pd.DataFrame(np.random.rand(15,2)))
hdf

<class 'pandas.io.pytables.HDFStore'>
File path: trip_data_1a.ziptrip_data_1a.h5

In [92]:
print(hdf.info())

<class 'pandas.io.pytables.HDFStore'>
File path: trip_data_1a.ziptrip_data_1a.h5
/data                 frame_table  (typ->appendable,nrows->5000000,ncols->14,indexers->[index],dc->[trip_distance])
/data/t1              frame        (shape->[15,2])                                                                 
/tables/t1            frame        (shape->[20,5])                                                                 
/tables/t2            frame        (shape->[10,3])                                                                 


In [93]:
hdf.close()

In [94]:
#release memory
del df

### Read subsets of data from disk using HDF5

If we have a really large file that cannot fit into RAM, but has been converted into an HDF5 format, we can read into memory only portions of it that are of interest to us an to our immediate processing needs. 

We can bring into RAM only selected columns, and/or rows that satisfy a certain criterion. We can do this is if the HDF5 file is searchable, and we can specify filter constraints during file read I/O on the indexed columns.

In [95]:
df = pd.read_hdf(file_path + 'trip_data_1a.h5', '/data', columns=['medallion','pickup_datetime', 'trip_distance'], where='trip_distance>10.0')
df.head()

,medallion,pickup_datetime,trip_distance
7,0B57B9633A2FECD3D3B1944AFC7471CF,2013-01-07 12:39:18,10.7
20,D8AADA4E722C87C10E609654612630DD,2013-01-10 23:41:51,17.5
28,8E189DABE265CC03FEF4BFC695B32559,2013-01-01 14:02:01,10.2
38,CD9DEF073BAB75B8B36015D85FD3F777,2013-01-05 03:20:28,11.2
46,8F9B593AE29F10B40EBEA0BBDEDD249F,2013-01-07 17:38:07,10.3


In [96]:
len(df)

244134

Notice that we imported a subset of columns above, based on a condition specified by the *where* argument.

**Exercise:** Write a query that reads in data from the above file where the trip distance is greater than 5 miles and smaller than 15 miles.

In [99]:
df = pd.read_hdf(file_path + 'trip_data_1a.h5', '/data', columns=['medallion','pickup_datetime', 'trip_distance'], where='(trip_distance>5.0) & (trip_distance<15.0)')
df.head()
# del df

,medallion,pickup_datetime,trip_distance
7,0B57B9633A2FECD3D3B1944AFC7471CF,2013-01-07 12:39:18,10.7
28,8E189DABE265CC03FEF4BFC695B32559,2013-01-01 14:02:01,10.2
30,F1EF8290A54338B1396DD98E38E09143,2013-01-05 08:16:58,6.5
32,33A0B414EB87DB2538CB5929BF1EEE0D,2013-01-05 17:34:11,5.2
34,7E3256C342CAFB3C23D3ABB98B9F3FB6,2013-01-05 03:00:46,5.5


In [101]:
print(len(df))
# del df

569175


**Exercise:** Write a query that reads in data from the above file where the number of passengers is greater than 5.

In [108]:
df = pd.read_hdf(file_path + 'trip_data_1a.h5', '/data', columns=['medallion','pickup_datetime', 'passenger_count'])
df = df[df['passenger_count'] > 5.0]
print(len(df))
# del df


169136


In [119]:
del df

You'll notice an error.

This is because the *passenger_count* filed is not indexed and is not searchable. 

**Exercise:** Re-write the trip_data_1a file into HDF5, enabling the search for *passenger_count*, then perform the above query again.

In [117]:
df = pd.read_hdf(file_path + 'trip_data_1a.h5', '/data', columns=['medallion','pickup_datetime', 'passenger_count'])
df = df[df['passenger_count'] > 5.0]
print(len(df))
# del df

**Exercise:** Read data from the above file, where the index number is between 10000 and 20000.

In [125]:
df = pd.read_hdf(file_path + 'trip_data_1a.h5', '/data', columns=['medallion','pickup_datetime', 'passenger_count'],
                start=10000, stop=20000)
print(len(df))
del df

10000


In [136]:
import parquet as pq

In [132]:
df = pd.read_csv(z.open('trip_data_1a.csv'), infer_datetime_format=True, parse_dates=['pickup_datetime', 'dropoff_datetime'])

C:\Users\johnl\AppData\Local\Temp\ipykernel_21196\4220178714.py:1: FutureWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df = pd.read_csv(z.open('trip_data_1a.csv'), infer_datetime_format=True, parse_dates=['pickup_datetime', 'dropoff_datetime'])


In [133]:
df.to_parquet('trip_data_1a.parquet')

In [138]:
import pyarrow.parquet as pq


In [139]:
table = pq.read_table('trip_data_1a.parquet')
table

pyarrow.Table
medallion: string
hack_license: string
vendor_id: string
rate_code: int64
store_and_fwd_flag: string
pickup_datetime: timestamp[ns]
dropoff_datetime: timestamp[ns]
passenger_count: int64
trip_time_in_secs: int64
trip_distance: double
pickup_longitude: double
pickup_latitude: double
dropoff_longitude: double
dropoff_latitude: double
----
medallion: [["89D227B655E5C82AECF13C3F540D4CF4","0BD7C8F5BA12B88E0B67BED28BEA73D8","0BD7C8F5BA12B88E0B67BED28BEA73D8","DFD2202EE08F7A8DC9A57B02ACB81FE2","DFD2202EE08F7A8DC9A57B02ACB81FE2",...,"7CC2C787A937EE1F874D2FFD6BBCD2F0","5FDB05E688DA0BB1D00BB8D10FD76F02","2218C94720E89EB1C6630E09C46EA8BC","71A788D349C8206115457A1574A006A2","73120D33FF179115E5E2E6C3177EAB77"],["D9598D121715B456C101D5CF72DEF513","37FBBEFC47751F81BAE4BA8394FE45A8","EB2020681F1B1B72375923FD0F8A73C6","3EAD22178DDDC1F91CD43C6B4A24E843","AB9097E0AE735D1482B3D350A8AD1AA0",...,"311388FEEE44CE52E5583E1091117E0A","874B24DA5769D8B22F4136FF55361C45","982E3D7F9FF185D430FD2E1701A7

# Out-of-core processing with the Dask DataFrame

### From now on we will pretend that our dataset is > RAM or large enough to take up too much of our RAM. Working with an actual dataset that is > RAM, will slow down all the examples in class considerably, but this is left as an exercise for you to perform out of class.

[`Dask`](https://dask.readthedocs.org/en/latest/) is a parallel computing library, designed to enable out-of-core processing. This means that most of the dataset can continue sitting on the hard disk while you are processing it, and it does not need to be read in it's entirety.

Dask has a DataFrame class which functions just like the pandas counterpart; however, the **Dask DataFrame does not have all the functionality of a pandas DataFrame**.

Dask can read large datasets from multiple file types.

We will read the same data from a csv and HDF5 files and compare the out-of-core performance efficiencies of the two file types.

We load this data into a dask dataframe using the `read_csv` function.  This has the exact same signature as `pandas.read_csv`.

In [126]:
#file_path = r'C:\datasets\teaching\datasets\\'

import pandas as pd
import numpy as np
import datetime as dt
import dask.dataframe as dd

In [ ]:
df = pd.read_csv(z.open('trip_data_1a.csv'), infer_datetime_format=True, parse_dates=['pickup_datetime', 'dropoff_datetime'])

In [145]:
file_path = "trip_data_1a.zip\"

SyntaxError: unterminated string literal (detected at line 1); perhaps you escaped the end quote? (686895117.py, line 1)

In [148]:
import dask.dataframe as dd

%time ddf_csv = dd.read_csv('trip_data_1a.csv', \
                 parse_dates=['pickup_datetime', 'dropoff_datetime'])
ddf_csv.head()

CPU times: total: 15.6 ms
Wall time: 38.3 ms


,medallion,hack_license,vendor_id,rate_code,store_and_fwd_flag,pickup_datetime,dropoff_datetime,passenger_count,trip_time_in_secs,trip_distance,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude
0,89D227B655E5C82AECF13C3F540D4CF4,BA96DE419E711691B9445D6A6307C170,CMT,1,N,2013-01-01 15:11:48,2013-01-01 15:18:10,4,382,1.0,-73.978165,40.757977,-73.989838,40.751171
1,0BD7C8F5BA12B88E0B67BED28BEA73D8,9FD8F69F0804BDB5549F40E9DA1BE472,CMT,1,N,2013-01-06 00:18:35,2013-01-06 00:22:54,1,259,1.5,-74.006683,40.731781,-73.994499,40.750660
2,0BD7C8F5BA12B88E0B67BED28BEA73D8,9FD8F69F0804BDB5549F40E9DA1BE472,CMT,1,N,2013-01-05 18:49:41,2013-01-05 18:54:23,1,282,1.1,-74.004707,40.737770,-74.009834,40.726002
3,DFD2202EE08F7A8DC9A57B02ACB81FE2,51EE87E3205C985EF8431D850C786310,CMT,1,N,2013-01-07 23:54:15,2013-01-07 23:58:20,2,244,0.7,-73.974602,40.759945,-73.984734,40.759388
4,DFD2202EE08F7A8DC9A57B02ACB81FE2,51EE87E3205C985EF8431D850C786310,CMT,1,N,2013-01-07 23:25:03,2013-01-07 23:34:24,1,560,2.1,-73.976250,40.748528,-74.002586,40.747868


In [149]:
ddf_csv.dtypes

medallion             string[pyarrow]
hack_license          string[pyarrow]
vendor_id             string[pyarrow]
rate_code                       int64
store_and_fwd_flag    string[pyarrow]
pickup_datetime        datetime64[ns]
dropoff_datetime       datetime64[ns]
passenger_count                 int64
trip_time_in_secs               int64
trip_distance                 float64
pickup_longitude              float64
pickup_latitude               float64
dropoff_longitude             float64
dropoff_latitude              float64
dtype: object

In [180]:
%time ddf_hdf = dd.read_hdf('trip_data_1a.ziptrip_data_1a.h5', '/data')
ddf_hdf.head()

CPU times: total: 31.2 ms
Wall time: 25.9 ms


,medallion,hack_license,vendor_id,rate_code,store_and_fwd_flag,pickup_datetime,dropoff_datetime,passenger_count,trip_time_in_secs,trip_distance,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude
0,89D227B655E5C82AECF13C3F540D4CF4,BA96DE419E711691B9445D6A6307C170,CMT,1,N,2013-01-01 15:11:48,2013-01-01 15:18:10,4,382,1.0,-73.978165,40.757977,-73.989838,40.751171
1,0BD7C8F5BA12B88E0B67BED28BEA73D8,9FD8F69F0804BDB5549F40E9DA1BE472,CMT,1,N,2013-01-06 00:18:35,2013-01-06 00:22:54,1,259,1.5,-74.006683,40.731781,-73.994499,40.750660
2,0BD7C8F5BA12B88E0B67BED28BEA73D8,9FD8F69F0804BDB5549F40E9DA1BE472,CMT,1,N,2013-01-05 18:49:41,2013-01-05 18:54:23,1,282,1.1,-74.004707,40.737770,-74.009834,40.726002
3,DFD2202EE08F7A8DC9A57B02ACB81FE2,51EE87E3205C985EF8431D850C786310,CMT,1,N,2013-01-07 23:54:15,2013-01-07 23:58:20,2,244,0.7,-73.974602,40.759945,-73.984734,40.759388
4,DFD2202EE08F7A8DC9A57B02ACB81FE2,51EE87E3205C985EF8431D850C786310,CMT,1,N,2013-01-07 23:25:03,2013-01-07 23:34:24,1,560,2.1,-73.976250,40.748528,-74.002586,40.747868


In [152]:
ddf_hdf.dtypes

medallion             string[pyarrow]
hack_license          string[pyarrow]
vendor_id             string[pyarrow]
rate_code                       int64
store_and_fwd_flag    string[pyarrow]
pickup_datetime        datetime64[ns]
dropoff_datetime       datetime64[ns]
passenger_count                 int64
trip_time_in_secs               int64
trip_distance                 float64
pickup_longitude              float64
pickup_latitude               float64
dropoff_longitude             float64
dropoff_latitude              float64
dtype: object

Calling *.compute()* on a Dask DataFrame, converts the returned data into a pandas DataFrame.

### Compare CSV to HDF5 speeds

We do a simple computation that requires reading a column of our dataset and compare performance between CSV files and our newly created HDF5 file.

In [153]:
%time ddf_csv.passenger_count.sum().compute()

CPU times: total: 15.7 s
Wall time: 8.68 s


np.int64(8395665)

In [154]:
%time ddf_hdf.passenger_count.sum().compute()

CPU times: total: 10.5 s
Wall time: 11.3 s


np.int64(8395665)

In [155]:
%time ddf_csv[['vendor_id','passenger_count']].groupby('vendor_id').count().compute()

CPU times: total: 17.5 s
Wall time: 10.6 s


,passenger_count
vendor_id,
CMT,2567259
VTS,2432741


In [156]:
%time ddf_hdf[['vendor_id','passenger_count']].groupby('vendor_id').count().compute()

CPU times: total: 10.8 s
Wall time: 10.8 s


,passenger_count
vendor_id,
CMT,2567259
VTS,2432741


In [157]:
ddf_hdf.head()

,medallion,hack_license,vendor_id,rate_code,store_and_fwd_flag,pickup_datetime,dropoff_datetime,passenger_count,trip_time_in_secs,trip_distance,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude
0,89D227B655E5C82AECF13C3F540D4CF4,BA96DE419E711691B9445D6A6307C170,CMT,1,N,2013-01-01 15:11:48,2013-01-01 15:18:10,4,382,1.0,-73.978165,40.757977,-73.989838,40.751171
1,0BD7C8F5BA12B88E0B67BED28BEA73D8,9FD8F69F0804BDB5549F40E9DA1BE472,CMT,1,N,2013-01-06 00:18:35,2013-01-06 00:22:54,1,259,1.5,-74.006683,40.731781,-73.994499,40.750660
2,0BD7C8F5BA12B88E0B67BED28BEA73D8,9FD8F69F0804BDB5549F40E9DA1BE472,CMT,1,N,2013-01-05 18:49:41,2013-01-05 18:54:23,1,282,1.1,-74.004707,40.737770,-74.009834,40.726002
3,DFD2202EE08F7A8DC9A57B02ACB81FE2,51EE87E3205C985EF8431D850C786310,CMT,1,N,2013-01-07 23:54:15,2013-01-07 23:58:20,2,244,0.7,-73.974602,40.759945,-73.984734,40.759388
4,DFD2202EE08F7A8DC9A57B02ACB81FE2,51EE87E3205C985EF8431D850C786310,CMT,1,N,2013-01-07 23:25:03,2013-01-07 23:34:24,1,560,2.1,-73.976250,40.748528,-74.002586,40.747868


**Exercise:** Construct a groupby query that groups the data by driver (medallion) and sums up all the trips travelled, listing the top 10. Do this using csv and hdf files and compare the execution times.

In [160]:
%time ddf_csv[['medallion','trip_time_in_secs']].groupby('medallion').sum().compute()

CPU times: total: 16.1 s
Wall time: 9.05 s


,trip_time_in_secs
medallion,
000318C2E3E6381580E5C99910A60668,281340
002B4CFC5B8920A87065FC131F9732D1,314160
002E3B405B6ABEA23B6305D3766140F1,303840
0030AD2648D81EE87796445DB61FCF20,188460
0035520A854E4F2769B37DAF5357426F,166920
...,...
CE40221F964C828384657FFD32F2885E,28192
D05213C123A9D151078C5689599860AC,0
DD65C3CD88CF99F9CE617B2D4C6D4DF1,0


In [159]:
%time ddf_hdf[['medallion','trip_time_in_secs']].groupby('medallion').sum().compute()

CPU times: total: 11.6 s
Wall time: 11 s


,trip_time_in_secs
medallion,
000318C2E3E6381580E5C99910A60668,281340
00153E36140C5B2A84EA308F355A7925,42900
002B4CFC5B8920A87065FC131F9732D1,314160
002E3B405B6ABEA23B6305D3766140F1,303840
0030AD2648D81EE87796445DB61FCF20,188460
...,...
ADB100692A33F2C1EA3096541F410838,112880
CE40221F964C828384657FFD32F2885E,28192
D05213C123A9D151078C5689599860AC,0


# Sampling

Sometimes when you have an enormous amount of data, you do not need to visualise or compute all of it. As long as you have out-of-core processing capabilities, it is often enough just to sample the data instead of working with all of it.

**Let's do some more processing of our data, but this time using the sampling strategy to handle the data size challenge,  and let's observe performance differences between accessing CSV and HDF5 files.**

### Density Map of NYC Rides

We'll use Bokeh to plot out a density map of all start and end locations for all taxi rides.

Get sample of pickup and dropoff locations

First we'll pick out a few columns and use the `sample` method to pick out a random sample of 1% of all rides.  We'll then call `compute` on this data to convert the results into in-memory Pandas DataFrames that we can hand off to Bokeh for visualisation.

We'll track our progress with the `ProgressBar` diagnostic.

In [161]:
from dask.diagnostics import ProgressBar

In [162]:
sample = ddf_csv.sample(frac=0.01)
print(len(sample))
pickup = sample[['pickup_latitude', 'pickup_longitude']]
with ProgressBar():
    result = pickup.compute()  # compute to smaller Pandas DataFrame

49999
[########################################] | 100% Completed | 10.29 s


In [163]:
len(result)

49999

In [164]:
from bokeh.plotting import figure, show, output_notebook
output_notebook()

Loading BokehJS ...

In [165]:
p = figure(title="Pickup Locations")
p.scatter(result.pickup_longitude, result.pickup_latitude, size=3, alpha=0.2)
show(p)

In [166]:
sample = ddf_hdf.sample(frac=0.01)
pickup = sample[['pickup_latitude', 'pickup_longitude']]
with ProgressBar():
    result = pickup.compute()  # compute to smaller Pandas DataFrame

[########################################] | 100% Completed | 14.28 s


In [167]:
p = figure(title="Pickup Locations")
p.scatter(result.pickup_longitude, result.pickup_latitude, size=3, alpha=0.2)
show(p)

#  Store text/string data more efficiently with categoricals

As it turns out, a considerable portion of access time is consumed by accessing string data. This is because Pandas represents text with the **object dtype** which holds a normal Python string. 

Object dtypes are usually the main cause of slow code because **object dtypes run at Python speeds, not at Pandas’ normal C speeds**.

Pandas **categoricals** are a new and powerful feature that **encodes categorical data into a numerical representation** so that we can leverage Pandas’ fast C code on this kind of text data.

Given this, there are some strategies we can use in order to encode string data more efficiently which will **improve performance**.

The strategy will demonstrate how to convert string data into categoricals. *This works very well when the data does not have a large number of distinct string values for a given column*.

This takes a bit more time up front, but results in better performance.

More on categoricals at the [pandas docs](http://pandas-docs.github.io/pandas-docs-travis/categorical.html).

In [173]:
ddf_hdf.head()

,medallion,hack_license,vendor_id,rate_code,store_and_fwd_flag,pickup_datetime,dropoff_datetime,passenger_count,trip_time_in_secs,trip_distance,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude
0,89D227B655E5C82AECF13C3F540D4CF4,BA96DE419E711691B9445D6A6307C170,CMT,1,N,2013-01-01 15:11:48,2013-01-01 15:18:10,4,382,1.0,-73.978165,40.757977,-73.989838,40.751171
1,0BD7C8F5BA12B88E0B67BED28BEA73D8,9FD8F69F0804BDB5549F40E9DA1BE472,CMT,1,N,2013-01-06 00:18:35,2013-01-06 00:22:54,1,259,1.5,-74.006683,40.731781,-73.994499,40.750660
2,0BD7C8F5BA12B88E0B67BED28BEA73D8,9FD8F69F0804BDB5549F40E9DA1BE472,CMT,1,N,2013-01-05 18:49:41,2013-01-05 18:54:23,1,282,1.1,-74.004707,40.737770,-74.009834,40.726002
3,DFD2202EE08F7A8DC9A57B02ACB81FE2,51EE87E3205C985EF8431D850C786310,CMT,1,N,2013-01-07 23:54:15,2013-01-07 23:58:20,2,244,0.7,-73.974602,40.759945,-73.984734,40.759388
4,DFD2202EE08F7A8DC9A57B02ACB81FE2,51EE87E3205C985EF8431D850C786310,CMT,1,N,2013-01-07 23:25:03,2013-01-07 23:34:24,1,560,2.1,-73.976250,40.748528,-74.002586,40.747868


We will categorise the 'vendor_id' and 'store_and_fwd_flag' columns.



In [176]:
# Categorize data
ddf_hdf = ddf_hdf.categorize(columns=['vendor_id','store_and_fwd_flag'])

In [172]:
ddf_hdf

,medallion,hack_license,vendor_id,rate_code,store_and_fwd_flag,pickup_datetime,dropoff_datetime,passenger_count,trip_time_in_secs,trip_distance,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude
npartitions=5,,,,,,,,,,,,,,
,string,string,category[known],int64,category[known],datetime64[ns],datetime64[ns],int64,int64,float64,float64,float64,float64,float64
,...,...,...,...,...,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
,...,...,...,...,...,...,...,...,...,...,...,...,...,...
,...,...,...,...,...,...,...,...,...,...,...,...,...,...


In [181]:
ddf_hdf = ddf_hdf.astype({col: 'object' for col in ddf_hdf.select_dtypes('string').columns})
ddf_hdf.to_hdf('trip_data_1a_categorical.h5', '/data', format='table', mode='w', data_columns=['trip_distance'])

['trip_data_1a_categorical.h5',
 'trip_data_1a_categorical.h5',
 'trip_data_1a_categorical.h5',
 'trip_data_1a_categorical.h5',
 'trip_data_1a_categorical.h5']

You should now notice that about a 25% reduction in disk space has been realized.

### Let's compare again our categorised HDF5, with the previous execution times:

(This may run slower on your computer depending on how full your RAM is from the previous datasets - try restarting the kernel and comparing the execution times from this point on without performing all the previous steps)

In [182]:
import pandas as pd
import datetime as dt
import dask.dataframe as dd
from dask.diagnostics import ProgressBar
from bokeh.plotting import figure, show, output_notebook
#import castra as c
output_notebook()

Loading BokehJS ...

In [183]:
import matplotlib.pyplot as plt
import numpy as np
import time
import seaborn as sns
from pylab import rcParams

%matplotlib inline
rcParams['figure.figsize'] = 20, 10
rcParams['font.size'] = 20

rcParams['figure.dpi'] = 350
rcParams['lines.linewidth'] = 2
rcParams['axes.facecolor'] = 'white'
rcParams['patch.edgecolor'] = 'white'
rcParams['font.family'] = 'StixGeneral'

In [ ]:
#file_path = r''

In [184]:
ddf_csv = dd.read_csv('trip_data_1a.csv', parse_dates=['pickup_datetime', 'dropoff_datetime'])
ddf_hdf = dd.read_hdf('trip_data_1a_categorical.h5', '/data')

In [185]:
ddf_hdf.head()

,medallion,hack_license,vendor_id,rate_code,store_and_fwd_flag,pickup_datetime,dropoff_datetime,passenger_count,trip_time_in_secs,trip_distance,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude
0,89D227B655E5C82AECF13C3F540D4CF4,BA96DE419E711691B9445D6A6307C170,CMT,1,N,2013-01-01 15:11:48,2013-01-01 15:18:10,4,382,1.0,-73.978165,40.757977,-73.989838,40.751171
1,0BD7C8F5BA12B88E0B67BED28BEA73D8,9FD8F69F0804BDB5549F40E9DA1BE472,CMT,1,N,2013-01-06 00:18:35,2013-01-06 00:22:54,1,259,1.5,-74.006683,40.731781,-73.994499,40.750660
2,0BD7C8F5BA12B88E0B67BED28BEA73D8,9FD8F69F0804BDB5549F40E9DA1BE472,CMT,1,N,2013-01-05 18:49:41,2013-01-05 18:54:23,1,282,1.1,-74.004707,40.737770,-74.009834,40.726002
3,DFD2202EE08F7A8DC9A57B02ACB81FE2,51EE87E3205C985EF8431D850C786310,CMT,1,N,2013-01-07 23:54:15,2013-01-07 23:58:20,2,244,0.7,-73.974602,40.759945,-73.984734,40.759388
4,DFD2202EE08F7A8DC9A57B02ACB81FE2,51EE87E3205C985EF8431D850C786310,CMT,1,N,2013-01-07 23:25:03,2013-01-07 23:34:24,1,560,2.1,-73.976250,40.748528,-74.002586,40.747868


Compare with previous execution runtimes:

In [186]:
%time ddf_hdf.passenger_count.sum().compute()

CPU times: total: 10 s
Wall time: 10.1 s


np.int64(8395665)

In [187]:
%time len(ddf_hdf)

CPU times: total: 10.8 s
Wall time: 11 s


5000000

In [188]:
%time ddf_hdf[['vendor_id','passenger_count']].groupby('vendor_id').count().compute()

CPU times: total: 12.8 s
Wall time: 12.4 s


,passenger_count
vendor_id,
CMT,2567259
VTS,2432741


In [189]:
sample = ddf_hdf.sample(frac=0.01)
pickup = sample[['pickup_latitude', 'pickup_longitude']]

with ProgressBar():
    result = pickup.compute()  # compute to smaller Pandas DataFrame

[########################################] | 100% Completed | 14.60 s


To fully appreciate and highlight the performance improvement of using categoricals, we will perform queries on categorical data that is in RAM.

Let's consider the medallion column which has not been categorised yet. Let's load it into RAM and perform a query on the original:


In [190]:
df = ddf_hdf[['medallion','trip_distance']].compute()

**Exercise:** Perform a groupby operation on this new dataframe and time the operation that groups the dataframe by medallion and sums the trip_distances for each medallion, and sorts the results in a descending order, showing the top ten medallions (taxis) by total distance travelled. 

**Exercise:** Convert the medallion columns into a category data type: 

**Exercise:** Perform the same group by operation as above. Time the execution and compare.

In [ ]:
del ddf_csv
del ddf_hdf

# More Efficient On-Disk Storage Combined with Columnar and Compression


We saw the significant speed improvement realised by resorting to HDF5. The disadvantage with HDF5 is that all the column data is stored together in contiguous blocks, which means that it is inefficient when a dataset comprises of numerous columns and frequently, only subsets of columns must be accessed.  

Therefore, there are more performance improvements that we can realise.

-----------------------------

Good on-disk storage for DataFrames generally has the following characteristics

1.  Binary storage format:  We don't store values as text that needs to be parsed as integers.  This rules out CSV and JSON.
2.  Categorical support:  Text is hard in Python.  Pandas Categoricals can speed this up when the text is often repeated.
3.  **Compression:  Light compression can often increase data bandwidth from disk.  Difficult to do right and depends largely on your data characteristics.**
2.  **Column Stores:  We store individual columns separately and in chunks so that queries on just a few columns can load a small fraction of the dataset.  This rules out typical use of HDF5.**

###   bcolz Project 

> bcolz provides columnar, chunked data containers that can be compressed either in-memory and on-disk. Column storage allows for efficiently querying tables, as well as for cheap column addition and removal. It is based on NumPy, and uses it as the standard data container to communicate with bcolz objects, but it also comes with support for import/export facilities to/from HDF5/PyTables tables and pandas dataframes.

> bcolz objects are compressed by default not only for reducing memory/disk storage, but also to improve I/O speed. The compression process is carried out internally by Blosc, a high-performance, multithreaded meta-compressor that is optimized for binary data (although it works with text data just fine too).

> bcolz can also use numexpr internally (it does that by default if it detects numexpr installed) or dask so as to accelerate many vector and query operations (although it can use pure NumPy for doing so too). numexpr/dask can optimize the memory usage and use multithreading for doing the computations, so it is blazing fast. This, in combination with carray/ctable disk-based, compressed containers, can be used for performing out-of-core computations efficiently, but most importantly transparently.

> By using compression, you can deal with more data using the same amount of memory, which is very good on itself. But in case you are wondering about the price to pay in terms of performance, you should know that nowadays memory access is the most common bottleneck in many computational scenarios, and that CPUs spend most of its time waiting for data. Hence, having data compressed in memory can reduce the stress of the memory subsystem as well.

> Furthermore, columnar means that the tabular datasets are stored column-wise order, and this turns out to offer better opportunities to improve compression ratio. This is because data tends to expose more similarity in elements that sit in the same column rather than those in the same row, so compressors generally do a much better job when data is aligned in such column-wise order. In addition, when you have to deal with tables with a large number of columns and your operations only involve some of them, a columnar-wise storage tends to be much more effective because minimizes the amount of data that travels to CPU caches.

> So, the ultimate goal for bcolz is not only reducing the memory needs of large arrays/tables, but also making bcolz operations to go faster than using a traditional data container like those in NumPy or Pandas. That is actually already the case in some real-life scenarios (see the notebook above) but that will become pretty more noticeable in combination with forthcoming, faster CPUs integrating more cores and wider vector units.

> source: https://github.com/Blosc/bcolz

Blosc is a compression library that is often used with bcolz:

> Blosc (http://blosc.org) is a high performance compressor optimized for binary data. It has been designed to transmit data to the processor cache faster than the traditional, non-compressed, direct memory fetch approach via a memcpy() OS call.

> Blosc works well for compressing numerical arrays that contains data with relatively low entropy, like sparse data, time series, grids with regular-spaced values, etc.

> python-blosc a Python package that wraps Blosc. python-blosc supports Python 2.7 and 3.4 or higher versions.

> source: https://github.com/Blosc/python-blosc

# SQL Lite

Sometimes it makes sense to store large data in SQLite (or other fully fledged Database Management Systems like MySQL, Postgres, MonongoDB etc. who are running as a separate process ).

In [202]:
import sqlite3
import datetime as dt

Converting our dataset to SQLite can be done by chunking.


In [192]:
df = pd.read_csv('trip_data_1a.csv', infer_datetime_format=True, parse_dates=['pickup_datetime', 'dropoff_datetime'])

C:\Users\johnl\AppData\Local\Temp\ipykernel_21196\94438499.py:1: FutureWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df = pd.read_csv('trip_data_1a.csv', infer_datetime_format=True, parse_dates=['pickup_datetime', 'dropoff_datetime'])


In [193]:
df.head()

,medallion,hack_license,vendor_id,rate_code,store_and_fwd_flag,pickup_datetime,dropoff_datetime,passenger_count,trip_time_in_secs,trip_distance,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude
0,89D227B655E5C82AECF13C3F540D4CF4,BA96DE419E711691B9445D6A6307C170,CMT,1,N,2013-01-01 15:11:48,2013-01-01 15:18:10,4,382,1.0,-73.978165,40.757977,-73.989838,40.751171
1,0BD7C8F5BA12B88E0B67BED28BEA73D8,9FD8F69F0804BDB5549F40E9DA1BE472,CMT,1,N,2013-01-06 00:18:35,2013-01-06 00:22:54,1,259,1.5,-74.006683,40.731781,-73.994499,40.750660
2,0BD7C8F5BA12B88E0B67BED28BEA73D8,9FD8F69F0804BDB5549F40E9DA1BE472,CMT,1,N,2013-01-05 18:49:41,2013-01-05 18:54:23,1,282,1.1,-74.004707,40.737770,-74.009834,40.726002
3,DFD2202EE08F7A8DC9A57B02ACB81FE2,51EE87E3205C985EF8431D850C786310,CMT,1,N,2013-01-07 23:54:15,2013-01-07 23:58:20,2,244,0.7,-73.974602,40.759945,-73.984734,40.759388
4,DFD2202EE08F7A8DC9A57B02ACB81FE2,51EE87E3205C985EF8431D850C786310,CMT,1,N,2013-01-07 23:25:03,2013-01-07 23:34:24,1,560,2.1,-73.976250,40.748528,-74.002586,40.747868


In [194]:
df = df.set_index('pickup_datetime')
df = df.sort_index()

In [195]:
df.head()

,medallion,hack_license,vendor_id,rate_code,store_and_fwd_flag,dropoff_datetime,passenger_count,trip_time_in_secs,trip_distance,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude
pickup_datetime,,,,,,,,,,,,,
2013-01-01,68AFAB054782BAAA2EC999094A9C6D68,7264192937469CAED78B3E5D7B24BC2C,VTS,1,NaN,2013-01-01 00:10:00,1,600,1.92,-74.006355,40.739651,-73.981247,40.738007
2013-01-01,647E33016ADAD0E4DAC3D659FC82320B,9185CC39562D96646192AD3F16288DC4,VTS,1,NaN,2013-01-01 00:12:00,2,720,6.63,-73.929344,40.807728,-73.979935,40.740757
2013-01-01,256C6A3CA47F81497C1F5038438B54B2,8FA961982C3FEF4386FB05B4667A2580,VTS,1,NaN,2013-01-01 00:05:00,1,300,0.13,-73.969841,40.797359,-73.961899,40.812466
2013-01-01,6B129E331EDE126B4E9AE336E45F0643,79D0A7B682ABC516B837B997968767C8,VTS,1,NaN,2013-01-01 00:08:00,5,480,2.73,-73.973907,40.747868,-73.948898,40.776649
2013-01-01,0BC78FA149D8D784815895BD33F17EE1,E283306B8C94BC6AD197480FABBA1F34,VTS,1,NaN,2013-01-01 00:12:00,1,720,4.00,-73.982605,40.756966,-73.947342,40.796516


In [197]:
%time df.to_hdf('trip_data_1a_pickup_datetime_index.h5', '/data', format='table', mode='w', data_columns=['trip_distance'])

<timed eval>:1: FutureWarning: Starting with pandas version 3.0 all arguments of to_hdf except for the argument 'path_or_buf' will be keyword-only.


CPU times: total: 18.5 s
Wall time: 19.2 s


In [198]:
ddf_datetime_index_hdf = dd.read_hdf('trip_data_1a_pickup_datetime_index.h5', '/data')
ddf_datetime_index_hdf.head()

,medallion,hack_license,vendor_id,rate_code,store_and_fwd_flag,dropoff_datetime,passenger_count,trip_time_in_secs,trip_distance,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude
pickup_datetime,,,,,,,,,,,,,
2013-01-01,68AFAB054782BAAA2EC999094A9C6D68,7264192937469CAED78B3E5D7B24BC2C,VTS,1,<NA>,2013-01-01 00:10:00,1,600,1.92,-74.006355,40.739651,-73.981247,40.738007
2013-01-01,647E33016ADAD0E4DAC3D659FC82320B,9185CC39562D96646192AD3F16288DC4,VTS,1,<NA>,2013-01-01 00:12:00,2,720,6.63,-73.929344,40.807728,-73.979935,40.740757
2013-01-01,256C6A3CA47F81497C1F5038438B54B2,8FA961982C3FEF4386FB05B4667A2580,VTS,1,<NA>,2013-01-01 00:05:00,1,300,0.13,-73.969841,40.797359,-73.961899,40.812466
2013-01-01,6B129E331EDE126B4E9AE336E45F0643,79D0A7B682ABC516B837B997968767C8,VTS,1,<NA>,2013-01-01 00:08:00,5,480,2.73,-73.973907,40.747868,-73.948898,40.776649
2013-01-01,0BC78FA149D8D784815895BD33F17EE1,E283306B8C94BC6AD197480FABBA1F34,VTS,1,<NA>,2013-01-01 00:12:00,1,720,4.00,-73.982605,40.756966,-73.947342,40.796516


In [199]:
ddf_datetime_index_hdf.loc['2013-1-1'].compute()

MemoryError: Unable to allocate 30.5 MiB for an array with shape (4000000,) and data type uint64

In [203]:
chunksize = 10
day = 1
date = '2013-1-'
time = ' 23:59:59'
total = 0

start = dt.datetime.now()
conn = sqlite3.connect("trip_data_1a_a.sl3")
while (day <= 20):
    temp = ddf_datetime_index_hdf.loc[date + str(day) : date + str(day) + time].compute()
    temp.reset_index(inplace=True)
    temp.to_sql('TAXITRIPDATA', conn, if_exists='append', index=False)
    day += chunksize
    total += len(temp)
    print('{} seconds: completed {} rows inserted'.format((dt.datetime.now() - start).seconds, total))

31 seconds: completed 333876 rows inserted
48 seconds: completed 474322 rows inserted


In [217]:
len(df.loc['2013-01-01'])+len(df.loc['2013-1-11 00:00:00'])
# df.loc['2013-01-01 00:00:00']

333878

In [ ]:
df.head()

In [218]:
conn.close()

Or, alternatively if you have a csv that needs to be converted into a sql file, this can also be done by chunking:

In [ ]:
chunksize = 100000
total = 0
start = dt.datetime.now()

conn2 = sqlite3.connect(file_path + "trip_data_1a_2.sl3")
for df in pd.read_csv(file_path + "trip_data_1a.csv", chunksize=chunksize):
    df.to_sql('TAXITRIPDATA', conn2, if_exists='append', index=False)
    total += len(df)
    print('{} seconds: completed {} rows inserted'.format((dt.datetime.now() - start).seconds, total))


In [ ]:
conn2.close()

In [219]:
del df

There are modules like **odo** that also automate the conversion process from multiple data types. See [`Odo documentation`](http://odo.pydata.org/en/latest/uri.html)

Once a sqlite database file has been created, it can be opened and queried using the SQL language and out-of-core-processing within pandas as follows:

In [224]:
conn

In [226]:
conn = sqlite3.connect("trip_data_1a_a.sl3")
curs = conn.cursor()

In [227]:
df_sql = pd.read_sql_query('SELECT * FROM TAXITRIPDATA LIMIT 10', conn)
df_sql

,pickup_datetime,medallion,hack_license,vendor_id,rate_code,store_and_fwd_flag,dropoff_datetime,passenger_count,trip_time_in_secs,trip_distance,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude
0,2013-01-01 00:00:00,68AFAB054782BAAA2EC999094A9C6D68,7264192937469CAED78B3E5D7B24BC2C,VTS,1,None,2013-01-01 00:10:00,1,600,1.92,-74.006355,40.739651,-73.981247,40.738007
1,2013-01-01 00:00:00,647E33016ADAD0E4DAC3D659FC82320B,9185CC39562D96646192AD3F16288DC4,VTS,1,None,2013-01-01 00:12:00,2,720,6.63,-73.929344,40.807728,-73.979935,40.740757
2,2013-01-01 00:00:00,256C6A3CA47F81497C1F5038438B54B2,8FA961982C3FEF4386FB05B4667A2580,VTS,1,None,2013-01-01 00:05:00,1,300,0.13,-73.969841,40.797359,-73.961899,40.812466
3,2013-01-01 00:00:00,6B129E331EDE126B4E9AE336E45F0643,79D0A7B682ABC516B837B997968767C8,VTS,1,None,2013-01-01 00:08:00,5,480,2.73,-73.973907,40.747868,-73.948898,40.776649
4,2013-01-01 00:00:00,0BC78FA149D8D784815895BD33F17EE1,E283306B8C94BC6AD197480FABBA1F34,VTS,1,None,2013-01-01 00:12:00,1,720,4.00,-73.982605,40.756966,-73.947342,40.796516
5,2013-01-01 00:00:00,6F38771E2ED14F0847A840D6E1A270AB,FB0B6758E00519A6D8F793A480281FA7,VTS,1,None,2013-01-01 00:39:00,5,2340,3.94,-73.984016,40.744183,-74.004189,40.742508
6,2013-01-01 00:00:00,3D436CD3F76226A6F20C783FBB1C235A,E7869308480393458887DCF97DC47482,VTS,1,None,2013-01-01 00:14:00,1,840,11.16,-73.951073,40.778755,-73.816467,40.739040
7,2013-01-01 00:00:00,AB53FA653DC3A411CC60FC5E52F90A27,EFDC0E4E4E8A493FF7BA2AA622766CE2,VTS,1,None,2013-01-01 00:11:00,1,660,9.35,-73.871170,40.773914,-73.805054,40.681210
8,2013-01-01 00:00:00,E3ADD882D2B3BD1B1A583F1EF4A58BFC,1F14C43D3C8C222FD13F6537197055AD,VTS,1,None,2013-01-01 00:08:00,2,480,1.68,-73.981575,40.767632,-73.977737,40.757927
9,2013-01-01 00:00:00,8EE28DA9049CC06587FBBD00AFA9A221,AFB2CCDBF58CEFC7B5B05524A5112EA2,VTS,1,None,2013-01-01 00:13:00,5,780,4.87,-74.004646,40.721268,-73.956963,40.766376


In [228]:
start = dt.datetime.now()
df_sql = pd.read_sql_query('SELECT medallion, pickup_datetime, dropoff_datetime, passenger_count '
                           'FROM TAXITRIPDATA '
                           'WHERE passenger_count > 5 '
                           'LIMIT 30', conn)
print('Operation took {} seconds.'.format((dt.datetime.now() - start).seconds))
df_sql.head(100)

Operation took 0 seconds.


,medallion,pickup_datetime,dropoff_datetime,passenger_count
0,58B2330EB7CE4AA0C5DD6E0D03621062,2013-01-01 00:00:00,2013-01-01 00:23:00,6
1,5C7501130D5FBCA3C65B6C6A5554D164,2013-01-01 00:00:00,2013-01-01 00:11:00,6
2,04D107A378DC761FDA5E92362DCA24E3,2013-01-01 00:00:00,2013-01-01 00:17:00,6
3,D9598D121715B456C101D5CF72DEF513,2013-01-01 00:01:00,2013-01-01 00:06:00,6
4,C1FD73EA10ED14B94601137EC9BAD606,2013-01-01 00:01:00,2013-01-01 00:31:00,6
5,107A1D09ABA818132B57308136D05D8E,2013-01-01 00:02:00,2013-01-01 00:17:00,6
6,655E773C92FA446353D5C8B7416BE818,2013-01-01 00:02:00,2013-01-01 00:05:00,6
7,7B3CA0F2574FA2F444E7E5858BCA75A9,2013-01-01 00:02:00,2013-01-01 00:18:00,6
8,72EAFBA3FB9F0507C671AD713A622FB6,2013-01-01 00:02:00,2013-01-01 00:12:00,6
9,D7BE8487A5ED42E6A829A642B320EE4B,2013-01-01 00:02:00,2013-01-01 00:16:00,6


Let's time the operation:

In [229]:
start = dt.datetime.now()
df_sql = pd.read_sql_query('SELECT medallion, COUNT(*) as `num_journeys` '
                           'FROM TAXITRIPDATA '
                           'GROUP BY medallion '
                           'ORDER BY -num_journeys '
                           'LIMIT 30', conn)
print('Operation took {} seconds.'.format((dt.datetime.now() - start).seconds))
df_sql.head(100)

Operation took 0 seconds.


,medallion,num_journeys
0,A540871E513CDAB6DD0CA9398E97ECE9,104
1,C7BF056F5E2EF6D4CCF68EA4A23502A7,101
2,FE0B0ED2B86356C4282FC4454B56EAEE,101
3,7E1346F23960CC18D7D129FA28B63A75,100
4,67BB49360B9CB8EBC2C1F007A6AF85B4,99
5,2F2F160D8428A7757500371F980468D5,98
6,359577AA045D6E26572E3B92BFDC6E5C,97
7,8D94D22A89832673875F92B80ADB012E,97
8,7ECD9CFAE09251F893DD7EB2718E1B51,96
9,0ECC4D3D2025A987532E1E5B5B26A119,95


The performance is far from ideal.

It's possible to improve the access time by creating an index on a key column.

**WARNING:** This might take a bit of time... 

**Exercise:** Write the SQL statement that will create an index on the 'medallion' field':

In [ ]:
start = dt.datetime.now()
curs = conn.cursor()

# YOUR CODE HERE
# ==============

# ==============

curs.execute(sql)
print('Operation took {} seconds.'.format((dt.datetime.now() - start).seconds))

**Exercise:** Re-execute the SQL query from the above example and compare the execution time with the newly set up index:

**Exercise:** Create a new index on another variable. Experiment with the above query to see if the new index has improved the performance.

Ensure that the database connection is closed once finished. 

In [ ]:
conn.close()

# DuckDB

In [230]:
import duckdb

parquet_path = 'trip_data_1a.parquet'

start = dt.datetime.now()
df_sql = duckdb.sql(f"""
    SELECT medallion, pickup_datetime, dropoff_datetime, passenger_count
    FROM read_parquet('{parquet_path}')
    WHERE passenger_count > 5
    LIMIT 30
    """).df()

print(f"operation took {(dt.datetime.now() - start).total_seconds():.3f} seconds.")
df_sql.head(100)

operation took 0.031 seconds.


,medallion,pickup_datetime,dropoff_datetime,passenger_count
0,EE1513D432B07F7E0B5E2ED1EF629086,2013-01-13 04:30:00,2013-01-13 04:44:00,6
1,282E88327FA32E0C655B1216A91125A6,2013-01-13 04:24:00,2013-01-13 04:41:00,6
2,FD39403FDE46B6C753DDD6518A4365D7,2013-01-13 04:41:00,2013-01-13 04:46:00,6
3,2789D8398CBD60E51BF7D4BC78F3D7A9,2013-01-13 04:31:00,2013-01-13 04:48:00,6
4,A0DEAEC3D5592AE94B876356F12F8158,2013-01-13 04:37:00,2013-01-13 04:42:00,6
5,7AD442D7CFA8B1D9FB63C37B0BD0C45C,2013-01-13 04:33:00,2013-01-13 04:47:00,6
6,72427BE07CFC31814826BCE9EB92D8D5,2013-01-13 04:41:00,2013-01-13 04:46:00,6
7,4599D51610F6A40450D57FC19990DD55,2013-01-13 04:25:00,2013-01-13 04:45:00,6
8,6AF797A6A38F0E9C7BBC5CA381FCA41A,2013-01-13 04:14:00,2013-01-13 04:44:00,6
9,1F7AC0BAC6F096CF7332414564DD232B,2013-01-13 04:41:00,2013-01-13 04:45:00,6


## Summary of Strategies For Dealing with Inconvenient Datasets

When dataset > RAM, we have a number of options in order to move forward and realise computational efficiency:

1. Chunking the dataset.
2. Using out-of-core processing libraries
3. Convert datasets to more efficient encodings for disk I/O operations during out-of-core processing
4. Filter binary datasets and bring into RAM only subsets of data
5. Look to optimise file encodings with the particular options they each provide - ie string categorisation 
6. Combine the above strategies with sampling. Use out-of-core processing to downsample before pass off the smaller dataset to `pandas` to process quickly in-memory.
7. Set an index on important columns
8. Use SQLite, or interface with a full-blown database management instance that is running in its own separate process
9. If you can fit the data into memory, consider Polars

# Conclusion

Storage choices strongly impact performance.  

This becomes even more critical when the data is large.

Strategies for handling data > RAM were covered.

Key points:
- store large datasets in a binary format
- ensure that the binary format supports search/filtering on key columns
- categorise string columns for more efficient queries
- consider using Dask for out-of-core processing as well as other emerging libraries (Bcolz)
- consider converting your data into a SQLite format, or if needed, loading it up into a database management system (MySQL, Oracle, SQL Server, MongoDB, Postgres etc.)
- investigate how you can optimise your code
